In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from collections import Counter
import re
import time
import os
import json

# ====== 설정 ======
stopwords = {"the", "and", "for", "with", "that", "this", "from", "are", "was", "has", "have", "but", "our"}
base_year = 2025
start_week = 45  # 시작 주차
max_weeks = 52   # 연도별 최대 주차

# ====== 웹 드라이버 실행 ======
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

year = base_year
week = start_week

while True:
    week_url = f"https://huggingface.co/papers/week/{year}-W{week:02d}"
    print(f"\n🔹 Crawling {week_url} ...")
    driver.get(week_url)

    # 페이지 존재 여부 확인
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "article h3 a"))
        )
    except:
        print(f"❌ Not found or no articles: {week_url} -> 크롤링 종료")
        break

    # 폴더 생성
    folder = f"{year}-W{week:02d}"
    os.makedirs(folder, exist_ok=True)

    # 파일명 시작 번호: 연도(2자리) + 주차(2자리) + 001
    file_index = int(str(year)[-2:] + f"{week:02d}" + "001")

    # 아티클 링크 추출
    articles = driver.find_elements(By.CSS_SELECTOR, "article h3 a")
    article_urls = [a.get_attribute("href") for a in articles]

    for link in article_urls:
        driver.get(link)

        # 논문 제목
        try:
            paper_name = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "h1"))
            ).text.strip()
        except:
            paper_name = "Unknown_Title"

        # Abstract
        page_content = ""
        try:
            abstract_div = WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div.pb-8.pr-4.md\\:pr-16 > div"))
            )
            ps = abstract_div.find_elements(By.TAG_NAME, "p")
            if ps:
                page_content = "\n".join([p.text.strip() for p in ps])
            else:
                page_content = abstract_div.text.strip()
        except:
            page_content = ""

        # Upvote
        try:
            upvote_elem = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located((By.CSS_SELECTOR,
                    "section.pt-8 div.hidden.flex-wrap.items-start.gap-2.md\\:flex > div > div > a > div > div"
                ))
            )
            upvote_text = upvote_elem.text.strip()
            upvote_match = re.search(r"\d+", upvote_text)
            upvote = int(upvote_match.group()) if upvote_match else 0
        except:
            upvote = 0

        # GitHub 링크
        try:
            github_url = driver.find_element(By.XPATH, "//a[contains(@href,'github.com')]").get_attribute("href")
        except:
            github_url = ""

        huggingface_url = link

        # 태그 추출 (abstract 단어 상위 3개)
        words = re.findall(r'\b\w+\b', page_content.lower())
        filtered = [w for w in words if w not in stopwords and len(w) > 2]
        counter = Counter(filtered)
        tags = [tag for tag, _ in counter.most_common(3)]
        while len(tags) < 3:
            tags.append("")

        # metadata dictionary
        MetaData = {
            "papername": paper_name,
            "github_url": github_url,
            "huggingface_url": huggingface_url,
            "upvote": upvote,
            "tag1": tags[0],
            "tag2": tags[1],
            "tag3": tags[2]
        }

        # 파일명 및 경로
        doc_name = f"doc{file_index}.txt"
        file_path = os.path.join(folder, doc_name)

        # txt 파일 작성
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(f"page_content:\n{page_content}\n\nMetaData:\n")
            f.write(json.dumps(MetaData, ensure_ascii=False, indent=4))

        print(f"✅ Saved {file_path}")
        file_index += 1

    # 다음 주차로 증가
    week += 1
    if week > max_weeks:
        week = 1
        year += 1

driver.quit()
print("모든 아티클 저장 완료!")



🔹 Crawling https://huggingface.co/papers/week/2025-W45 ...
✅ Saved 2025-W45\doc2545001.txt
✅ Saved 2025-W45\doc2545002.txt
✅ Saved 2025-W45\doc2545003.txt
✅ Saved 2025-W45\doc2545004.txt
✅ Saved 2025-W45\doc2545005.txt
✅ Saved 2025-W45\doc2545006.txt
✅ Saved 2025-W45\doc2545007.txt
✅ Saved 2025-W45\doc2545008.txt
✅ Saved 2025-W45\doc2545009.txt
✅ Saved 2025-W45\doc2545010.txt
✅ Saved 2025-W45\doc2545011.txt
✅ Saved 2025-W45\doc2545012.txt
✅ Saved 2025-W45\doc2545013.txt
✅ Saved 2025-W45\doc2545014.txt
✅ Saved 2025-W45\doc2545015.txt
✅ Saved 2025-W45\doc2545016.txt
✅ Saved 2025-W45\doc2545017.txt
✅ Saved 2025-W45\doc2545018.txt
✅ Saved 2025-W45\doc2545019.txt
✅ Saved 2025-W45\doc2545020.txt
✅ Saved 2025-W45\doc2545021.txt
✅ Saved 2025-W45\doc2545022.txt
✅ Saved 2025-W45\doc2545023.txt
✅ Saved 2025-W45\doc2545024.txt
✅ Saved 2025-W45\doc2545025.txt
✅ Saved 2025-W45\doc2545026.txt
✅ Saved 2025-W45\doc2545027.txt
✅ Saved 2025-W45\doc2545028.txt
✅ Saved 2025-W45\doc2545029.txt
✅ Saved 2025

In [9]:
import os
import json

# 확인할 폴더들 (예: 2025-W45, 2025-W46 등)
folders = [f for f in os.listdir() if os.path.isdir(f) and f.startswith("2025-W")]

for folder in folders:
    print(f"\n🔹 Checking folder: {folder}")
    empty_count = 0

    # 폴더 안의 txt 파일 확인
    for filename in os.listdir(folder):
        if filename.endswith(".txt"):
            file_path = os.path.join(folder, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                content = f.read()
                
                # page_content 부분 추출
                if "page_content:\n" in content:
                    page_content = content.split("page_content:\n")[1].split("\n\nMetaData:")[0].strip()
                    if not page_content:
                        print(f"⚠️ Empty page_content: {file_path}")
                        empty_count += 1
                else:
                    print(f"❌ page_content not found: {file_path}")
                    empty_count += 1

    print(f"✅ Total empty page_content in {folder}: {empty_count}")



🔹 Checking folder: 2025-W45
⚠️ Empty page_content: 2025-W45\doc2545089.txt
✅ Total empty page_content in 2025-W45: 1

🔹 Checking folder: 2025-W46
✅ Total empty page_content in 2025-W46: 0


In [12]:
import os
import json
import re

# 체크할 최상위 폴더 (예: 현재 폴더)
base_dir = "."

# 폴더 패턴: 2025-W45, 2025-W46 ...
week_folders = [f for f in os.listdir(base_dir) if re.match(r"\d{4}-W\d{2}", f)]

for folder in week_folders:
    folder_path = os.path.join(base_dir, folder)
    txt_files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]

    for txt_file in txt_files:
        file_path = os.path.join(folder_path, txt_file)
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()

        # metadata 부분 추출
        meta_match = re.search(r"MetaData:\s*(\{.*\})", content, re.DOTALL)
        if not meta_match:
            print(f"⚠️ MetaData not found: {file_path}")
            continue

        try:
            metadata = json.loads(meta_match.group(1))
        except:
            print(f"⚠️ MetaData JSON parse error: {file_path}")
            continue

        # 빈 값 체크
        empty_fields = [k for k, v in metadata.items() if v == "" or v is None]
        if empty_fields:
            print(f"⚠️ Empty MetaData fields in {file_path}: {empty_fields}")


⚠️ Empty MetaData fields in .\2025-W45\doc2545006.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545008.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545013.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545014.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545015.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545016.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545017.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545020.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545021.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545023.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545026.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545028.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545030.txt: ['github_url']
⚠️ Empty MetaData fields in .\2025-W45\doc2545031.txt: ['github_url']
⚠️ Empty MetaData fi

In [15]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from collections import Counter
import re, time, os, json, random

# ====== 설정 ======
base_year = 2025
start_week = 45
wait_time = 10               # 페이지 렌더링 대기시간
max_retry_per_article = 5    # 아티클 단위 재시도
retry_click = 5              # 오른쪽 버튼 클릭 재시도
stopwords = {"the", "and", "for", "with", "that", "this", "from", "are", "was", "has", "have", "but", "our"}

# ====== 웹 드라이버 실행 ======
options = webdriver.ChromeOptions()
# options.add_argument("--headless=new") # 브라우저를 **화면 없이** 실행. 즉, 실제 창이 뜨지 않고 백그라운드에서 동작
# options.add_argument("--disable-gpu") # GPU 하드웨어 가속을 사용하지 않음. 주로 헤드리스 모드에서 안정성 위해 추가
# options.add_argument("--window-size=1920,1080")  # 브라우저 가상 창 크기를 지정. 화면 크기 기반 렌더링이나 스크린샷 필요 시 중요
# options.add_argument(f"--user-agent=Mozilla/5.0") # 웹 서버에 "브라우저 요청"임을 알리는 User-Agent 설정

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ====== 초기 주차 URL ======
week = start_week
week_url = f"https://huggingface.co/papers/week/{base_year}-W{week:02d}"
driver.get(week_url)

# ====== 파일명 시작 ======
file_index = int(str(base_year)[-2:] + f"{week:02d}" + "001")

while True:
    print(f"\n🔹 Crawling {week_url} ...")
    folder = f"{base_year}-W{week:02d}"
    os.makedirs(folder, exist_ok=True)

    # 랜덤 대기
    time.sleep(random.uniform(8, 15))

    # 아티클 링크 추출
    try:
        WebDriverWait(driver, wait_time).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "article h3 a"))
        )
        articles = driver.find_elements(By.CSS_SELECTOR, "article h3 a")
        article_urls = [a.get_attribute("href") for a in articles]
    except:
        print(f"❌ No articles found on {week_url}. 스킵")
        break

    for link in article_urls:
        for attempt in range(1, max_retry_per_article+1):
            try:
                driver.get(link)
                time.sleep(random.uniform(8, 12))  # 랜덤 대기

                # 논문 제목
                try:
                    paper_name = WebDriverWait(driver, wait_time).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "h1"))
                    ).text.strip()
                except:
                    paper_name = "Unknown_Title"

                # Abstract
                page_content = ""
                try:
                    abstract_div = WebDriverWait(driver, wait_time).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR, "div.pb-8.pr-4.md\\:pr-16 > div"))
                    )
                    ps = abstract_div.find_elements(By.TAG_NAME, "p")
                    page_content = "\n".join([p.text.strip() for p in ps]) if ps else abstract_div.text.strip()
                except:
                    page_content = ""

                # Upvote
                try:
                    upvote_elem = WebDriverWait(driver, wait_time).until(
                        EC.presence_of_element_located((By.CSS_SELECTOR,
                            "section.pt-8 div.hidden.flex-wrap.items-start.gap-2.md\\:flex > div > div > a > div > div"
                        ))
                    )
                    upvote_match = re.search(r"\d+", upvote_elem.text.strip())
                    upvote = int(upvote_match.group()) if upvote_match else 0
                except:
                    upvote = 0

                # GitHub 링크
                try:
                    github_url = driver.find_element(By.XPATH, "//a[contains(@href,'github.com')]").get_attribute("href")
                except:
                    github_url = ""

                huggingface_url = link

                # 태그 추출
                words = re.findall(r'\b\w+\b', page_content.lower())
                filtered = [w for w in words if w not in stopwords and len(w) > 2]
                counter = Counter(filtered)
                tags = [tag for tag, _ in counter.most_common(3)]
                while len(tags) < 3:
                    tags.append("")

                # metadata dictionary
                MetaData = {
                    "papername": paper_name,
                    "github_url": github_url,
                    "huggingface_url": huggingface_url,
                    "upvote": upvote,
                    "tag1": tags[0],
                    "tag2": tags[1],
                    "tag3": tags[2]
                }

                # 파일명 및 경로
                doc_name = f"doc{file_index}.txt"
                file_path = os.path.join(folder, doc_name)

                # txt 파일 작성
                with open(file_path, "w", encoding="utf-8") as f:
                    f.write(f"page_content:\n{page_content}\n\nMetaData:\n")
                    f.write(json.dumps(MetaData, ensure_ascii=False, indent=4))

                print(f"✅ Saved {file_path}")
                file_index += 1
                break  # 성공하면 재시도 종료

            except Exception as e:
                print(f"⚠️ 아티클 로딩 실패, 재시도 {attempt}/{max_retry_per_article}")
                time.sleep(10)  # 재시도 전 대기

    # ====== 다음 주 오른쪽 화살표 클릭 ======
    clicked = False
    for attempt in range(retry_click):
        try:
            next_btn = WebDriverWait(driver, wait_time).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR,
                    "body > div.flex.min-h-dvh.flex-col > main > div:nth-child(2) > section div.ml-6.flex.items-center.overflow-hidden a:nth-child(3)"
                ))
            )
            next_btn.click()
            time.sleep(random.uniform(8, 15))
            week_url = driver.current_url
            week += 1
            file_index = int(str(base_year)[-2:] + f"{week:02d}" + "001")
            clicked = True
            break
        except:
            print(f"⚠️ 오른쪽 버튼 클릭 실패, 재시도 {attempt+1}/{retry_click}")
            time.sleep(3)

    if not clicked:
        print("➡ 더 이상 오른쪽 버튼 없음 -> 최신 주차 도달, 크롤링 종료")
        break

driver.quit()
print("모든 아티클 저장 완료!")



🔹 Crawling https://huggingface.co/papers/week/2025-W45 ...
✅ Saved 2025-W45\doc2545001.txt
✅ Saved 2025-W45\doc2545002.txt
✅ Saved 2025-W45\doc2545003.txt
✅ Saved 2025-W45\doc2545004.txt
✅ Saved 2025-W45\doc2545005.txt
✅ Saved 2025-W45\doc2545006.txt
✅ Saved 2025-W45\doc2545007.txt
✅ Saved 2025-W45\doc2545008.txt
✅ Saved 2025-W45\doc2545009.txt
✅ Saved 2025-W45\doc2545010.txt
✅ Saved 2025-W45\doc2545011.txt
✅ Saved 2025-W45\doc2545012.txt
✅ Saved 2025-W45\doc2545013.txt
✅ Saved 2025-W45\doc2545014.txt
✅ Saved 2025-W45\doc2545015.txt
✅ Saved 2025-W45\doc2545016.txt
✅ Saved 2025-W45\doc2545017.txt
✅ Saved 2025-W45\doc2545018.txt
✅ Saved 2025-W45\doc2545019.txt
✅ Saved 2025-W45\doc2545020.txt
✅ Saved 2025-W45\doc2545021.txt
✅ Saved 2025-W45\doc2545022.txt
✅ Saved 2025-W45\doc2545023.txt
✅ Saved 2025-W45\doc2545024.txt
✅ Saved 2025-W45\doc2545025.txt
✅ Saved 2025-W45\doc2545026.txt
✅ Saved 2025-W45\doc2545027.txt
✅ Saved 2025-W45\doc2545028.txt
✅ Saved 2025-W45\doc2545029.txt
✅ Saved 2025